# **AI-Driven Phishing Email Detection Using NLP (Natural Language Processing)**

**Course:** B.Tech Artificial Intelligence & Machine Learning (AIML)  
**Project Title:** Semester Project - End-to-End Phishing Detection  
**Author:** Ayush  
**Language:** Python  
**Frameworks:** Pandas, NumPy, NLTK, Scikit-learn, Matplotlib, Seaborn, WordCloud  

---

## **1. Project Overview & Objective**
Phishing is a cyberattack where attackers send deceptive messages designed to trick victims into revealing sensitive information, executing malware, or authorizing unauthorized transactions. Phishing messages typically impersonate legitimate organizations.

**Objective:** Build a machine learning-based classification system using NLP techniques to detect whether an email is **Spam (Phishing)** or **Ham (Legitimate)**. This project implements a full machine learning workflow including data ingestion, cleaning, text preprocessing, exploratory data analysis (EDA), feature engineering, model training, performance evaluation, and visualization comparison.


## **2. Environment Setup & Dependency Import**
First, we import the required libraries. These libraries handle data manipulation (Pandas, NumPy), text preprocessing (NLTK), model training/evaluation (Scikit-learn), and data visualization (Matplotlib, Seaborn, WordCloud).


In [ ]:
import os
import re
import string
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from wordcloud import WordCloud
from collections import Counter

# Natural Language Toolkit
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.stem import WordNetLemmatizer

# Scikit-Learn Utilities and Models
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.svm import LinearSVC
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score, 
                             confusion_matrix, classification_report, roc_curve, auc)

# Ensure NLTK datasets are downloaded
nltk.download('stopwords', quiet=True)
nltk.download('wordnet', quiet=True)
nltk.download('punkt', quiet=True)
nltk.download('omw-1.4', quiet=True)

print('All libraries imported and resources downloaded successfully!')


## **3. Dataset Loading and Inspection**
We load the primary dataset `spam.csv`. We will programmatically inspect the dataset to understand its dimensions, retrieve column names, identify missing values, and analyze duplicate entries.


In [ ]:
# Load dataset with latin-1 encoding to handle special characters
df = pd.read_csv('enron_spam_data.csv')

# Inspect shape
print(f'Dataset Shape: {df.shape[0]} rows, {df.shape[1]} columns')
print('\nColumn names:', df.columns.tolist())
print('\nMissing values per column:\n', df.isnull().sum())
print('\nDuplicate records in dataset:', df.duplicated().sum())
print('\nClass distribution (labels):\n', df['v1'].value_counts())


## **4. Data Cleaning & Preprocessing**
### **Why clean data?**
Text data in the wild is messy. In this step, we perform:
1. **Dropping unnecessary columns:** Remove the unnamed empty columns.
2. **Null handling:** Drop missing records.
3. **De-duplication:** Remove duplicate records to prevent data leakage.
4. **Label Encoding:** Convert nominal target labels `ham` and `spam` to binary format (`0` and `1` respectively).
5. **Text Normalization:** Apply standard NLP techniques (lowercasing, HTML tag removal, URL removal, number removal, punctuation removal, tokenization, stopword removal, and WordNet lemmatization) to normalize text representations.

Let's write a comprehensive preprocessing pipeline and display the intermediate inputs.


In [ ]:
# 1. Remove unnecessary columns
df = df.iloc[:, [0, 1]]
df.columns = ['label', 'text']

# 2. Remove null values (if any)
df = df.dropna()

# 3. Remove duplicates
df = df.drop_duplicates().reset_index(drop=True)

# 4. Encode labels: ham -> 0, spam -> 1
df['label_bin'] = df['label'].map({'ham': 0, 'spam': 1})

print(f'Shape after cleaning: {df.shape[0]} rows, {df.shape[1]} columns')
print('Label mapping check: ham->0, spam->1')


### **Text Preprocessing Pipeline**
We write a dedicated function `preprocess_text` implementing the specified filters. We then apply it to our corpus and display sample text modifications.


In [ ]:
lemmatizer = WordNetLemmatizer()
stop_words = set(stopwords.words('english'))

def preprocess_text(text):
    if not isinstance(text, str):
        return ''
    # Lowercase text
    text = text.lower()
    # Remove HTML tags using regular expression
    text = re.sub(r'<[^>]+>', '', text)
    # Remove URLs using regular expression
    text = re.sub(r'https?://\S+|www\.\S+', '', text)
    # Remove numbers using regular expression
    text = re.sub(r'\d+', '', text)
    # Remove punctuation
    text = text.translate(str.maketrans('', '', string.punctuation))
    # Tokenization
    tokens = word_tokenize(text)
    # Remove stopwords and Lemmatize tokens
    cleaned_tokens = [lemmatizer.lemmatize(word) for word in tokens if word not in stop_words and len(word) > 1]
    return ' '.join(cleaned_tokens)

# Apply preprocessing
df['cleaned_text'] = df['text'].apply(preprocess_text)

# Display sample intermediate outputs
sample_indices = [0, 5, 8, 12, 15]
for idx in sample_indices:
    print(f'--- Record {idx} ({df.loc[idx, "label"]}) ---')
    print('Original : ', df.loc[idx, 'text'])
    print('Cleaned  : ', df.loc[idx, 'cleaned_text'])
    print()


## **5. Exploratory Data Analysis (EDA)**
In this section, we analyze the structural differences between Spam and Ham emails using visual charts:
1. **Class Distribution (Bar & Pie):** Visualize target class distribution.
2. **Email Character Length & Word Count Distributions:** Compare distributions by category.
3. **Top 20 Most Frequent Words:** Identify the most dominant words.
4. **WordClouds for Spam and Ham:** Display keyword density.


In [ ]:
sns.set_theme(style='whitegrid')

# Add length features
df['length'] = df['text'].apply(len)
df['word_count'] = df['text'].apply(lambda x: len(x.split()))

# Create figure subplots for class distributions
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# (a) Bar chart of class distribution
sns.countplot(x='label', data=df, palette='viridis', ax=axes[0])
axes[0].set_title('Class Distribution (Count)', fontsize=12, fontweight='bold')
axes[0].set_xlabel('Class')
axes[0].set_ylabel('Count')
for p in axes[0].patches:
    axes[0].annotate(f'{int(p.get_height())}', (p.get_x() + p.get_width() / 2., p.get_height()),
                ha='center', va='center', xytext=(0, 5), textcoords='offset points', fontweight='semibold')

# (b) Pie chart of class distribution
labels = df['label'].value_counts().index.map({'ham': 'Ham (Legitimate)', 'spam': 'Spam (Phishing)'})
sizes = df['label'].value_counts().values
axes[1].pie(sizes, labels=labels, autopct='%1.1f%%', startangle=140, colors=['#4ea8de', '#560bad'], 
        explode=(0, 0.1), textprops={'fontsize': 11, 'fontweight': 'bold'}, shadow=True)
axes[1].set_title('Proportion of Spam vs Ham Emails', fontsize=12, fontweight='bold')

plt.tight_layout()
plt.show()


### **Email Length & Word Count Distributions**
Legitimate emails and spam/phishing emails show different formatting habits. We plot histograms with kernel density estimates (KDE) to visualize character length and word count properties.


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# Length distribution
sns.histplot(data=df, x='length', hue='label', kde=True, bins=50, palette='mako', ax=axes[0], multiple='stack')
axes[0].set_title('Email Character Length Distribution by Class', fontsize=12, fontweight='bold')
axes[0].set_xlabel('Character Length')
axes[0].set_ylabel('Frequency')
axes[0].set_xlim(0, 300)

# Word count distribution
sns.histplot(data=df, x='word_count', hue='label', kde=True, bins=50, palette='crest', ax=axes[1], multiple='stack')
axes[1].set_title('Word Count Distribution by Class', fontsize=12, fontweight='bold')
axes[1].set_xlabel('Word Count')
axes[1].set_ylabel('Frequency')
axes[1].set_xlim(0, 60)

plt.tight_layout()
plt.show()


### **Word Frequency Analysis**
Let's identify the top 20 most frequent words in the cleaned text and generate word clouds.


In [ ]:
def get_top_words(texts, n=20):
    words = []
    for t in texts:
        words.extend(t.split())
    return pd.DataFrame(Counter(words).most_common(n), columns=['Word', 'Frequency'])

top_overall = get_top_words(df['cleaned_text'])

# Plot top 20 words
plt.figure(figsize=(10, 5))
sns.barplot(x='Frequency', y='Word', data=top_overall, palette='coolwarm')
plt.title('Top 20 Most Frequent Words (Overall cleaned text)', fontsize=14, fontweight='bold')
plt.xlabel('Frequency')
plt.ylabel('Word')
plt.tight_layout()
plt.show()


### **WordClouds for Spam vs Ham**
A WordCloud is a visual representation of text data, where the size of each word indicates its frequency or importance.


In [ ]:
spam_text = ' '.join(df[df['label'] == 'spam']['cleaned_text'])
ham_text = ' '.join(df[df['label'] == 'ham']['cleaned_text'])

fig, axes = plt.subplots(1, 2, figsize=(16, 8))

# Spam WordCloud
wc_spam = WordCloud(width=600, height=400, background_color='black', colormap='Reds', max_words=100).generate(spam_text)
axes[0].imshow(wc_spam, interpolation='bilinear')
axes[0].axis('off')
axes[0].set_title('WordCloud for Spam (Phishing) Emails', fontsize=14, fontweight='bold', pad=10)

# Ham WordCloud
wc_ham = WordCloud(width=600, height=400, background_color='black', colormap='GnBu', max_words=100).generate(ham_text)
axes[1].imshow(wc_ham, interpolation='bilinear')
axes[1].axis('off')
axes[1].set_title('WordCloud for Ham (Legitimate) Emails', fontsize=14, fontweight='bold', pad=10)

plt.tight_layout()
plt.show()


## **6. Feature Engineering & Train-Test Split**
### **Why Feature Engineering?**
Machine learning models require inputs to be numerical vectors. We compare two vectorization techniques:
1. **CountVectorizer:** Represents text as a bag-of-words where each feature value is the count of that term in a message.
2. **TF-IDF Vectorizer:** Scales term count by its inverse document frequency, highlighting words that carry distinct semantic information.

### **Train-Test Split**
We use **80% training data** to build models and **20% testing data** to evaluate them, with a random seed of `42` to ensure reproducibility.


In [ ]:
X = df['cleaned_text']
y = df['label_bin']

# Train-Test Split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print(f'Training Set Size: {X_train.shape[0]} samples')
print(f'Testing Set Size: {X_test.shape[0]} samples')

# Extract Features using both vectorizers
count_vect = CountVectorizer()
tfidf_vect = TfidfVectorizer(ngram_range=(1, 2), min_df=3, sublinear_tf=True)

X_train_count = count_vect.fit_transform(X_train)
X_test_count = count_vect.transform(X_test)

X_train_tfidf = tfidf_vect.fit_transform(X_train)
X_test_tfidf = tfidf_vect.transform(X_test)

print(f'Vocabulary size: {len(tfidf_vect.vocabulary_)} features')


### **Theoretical Comparison: Why TF-IDF performs better**
While CountVectorizer lists counts of words, TF-IDF divides count frequencies by document frequency. This means terms that occur uniformly across ham and spam documents (e.g., standard greetings, standard conversational verbs) receive lower weights, while specific terms (e.g., 'prize', 'winner', 'account', 'verify', 'urgent') receive higher weights. Mathematically, it resolves document-length bias and prevents sparse vocabulary dominance.


## **7. Machine Learning Models**
We train and compare five classification algorithms on the TF-IDF representation:
1. **Logistic Regression**
2. **Multinomial Naive Bayes**
3. **Random Forest**
4. **Decision Tree**
5. **Support Vector Machine (LinearSVC)**


In [ ]:
models = {
    'Logistic Regression': LogisticRegression(C=5.0, random_state=42),
    'Multinomial Naive Bayes': MultinomialNB(alpha=0.1),
    'Random Forest': RandomForestClassifier(n_estimators=200, random_state=42),
    'Decision Tree': DecisionTreeClassifier(random_state=42),
    'Support Vector Machine': LinearSVC(C=1.0, random_state=42)
}

results_tfidf = {}

for name, model in models.items():
    # Train
    model.fit(X_train_tfidf, y_train)
    # Predict
    y_pred = model.predict(X_test_tfidf)
    
    # Calculate metrics
    results_tfidf[name] = {
        'accuracy': accuracy_score(y_test, y_pred),
        'precision': precision_score(y_test, y_pred),
        'recall': recall_score(y_test, y_pred),
        'f1_score': f1_score(y_test, y_pred),
        'confusion_matrix': confusion_matrix(y_test, y_pred),
        'classification_report': classification_report(y_test, y_pred, output_dict=True)
    }
    
print('All models trained and evaluated on TF-IDF representation!')


## **8. Model Evaluation & Metrics**
We print classification reports and plot visual metrics for comparison.


In [ ]:
for name in results_tfidf:
    print(f'===================== {name} =====================')
    print(f'Accuracy  : {results_tfidf[name]["accuracy"]:.4%}')
    print(f'Precision : {results_tfidf[name]["precision"]:.4%}')
    print(f'Recall    : {results_tfidf[name]["recall"]:.4%}')
    print(f'F1-Score  : {results_tfidf[name]["f1_score"]:.4%}')
    print('\nClassification Report:\n')
    cr_df = pd.DataFrame(results_tfidf[name]['classification_report']).transpose()
    print(cr_df)
    print('\n')


### **Confusion Matrix Heatmaps**
Confusion matrices visually represent model predictions against actual labels. We plot them below.


In [ ]:
fig, axes = plt.subplots(3, 2, figsize=(12, 14))
axes = axes.flatten()
model_names = list(results_tfidf.keys())

for idx, name in enumerate(model_names):
    cm = results_tfidf[name]['confusion_matrix']
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[idx], cbar=False,
                xticklabels=['Ham', 'Spam'], yticklabels=['Ham', 'Spam'])
    axes[idx].set_title(f'{name} Confusion Matrix', fontsize=12, fontweight='bold')
    axes[idx].set_xlabel('Predicted')
    axes[idx].set_ylabel('True')

fig.delaxes(axes[5])  # Delete empty subplot
plt.tight_layout()
plt.show()


### **Performance Metric Comparison Graphs**
We plot accuracy, precision, recall, and F1-score comparisons across the models.


In [ ]:
metrics = ['accuracy', 'precision', 'recall', 'f1_score']
titles = ['Accuracy Comparison', 'Precision Comparison', 'Recall Comparison', 'F1-Score Comparison']
colors = ['Blues_r', 'Oranges_r', 'Greens_r', 'Purples_r']

fig, axes = plt.subplots(2, 2, figsize=(16, 12))
axes = axes.flatten()

for i, metric in enumerate(metrics):
    scores = [results_tfidf[m][metric] for m in model_names]
    sns.barplot(x=scores, y=model_names, palette=colors[i], ax=axes[i])
    axes[i].set_title(titles[i], fontsize=13, fontweight='bold')
    axes[i].set_xlabel(metric.capitalize())
    axes[i].set_ylabel('Models')
    axes[i].set_xlim(0.75, 1.0)
    for p in axes[i].patches:
        width = p.get_width()
        axes[i].text(width + 0.005, p.get_y() + p.get_height()/2., f'{width:.4f}', 
                     va='center', ha='left', fontweight='semibold')

plt.tight_layout()
plt.show()


### **Feature Importance & ROC Curves**
1. **Feature Importance:** Highlights the top terms contributing to decision tree splits in Random Forest.
2. **ROC Curves:** Measures the trade-off between True Positive Rate and False Positive Rate across classification thresholds.


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(18, 7))

# 1. Feature Importance (Random Forest)
rf_model = models['Random Forest']
importances = rf_model.feature_importances_
indices = np.argsort(importances)[::-1][:20]
feature_names = np.array(tfidf_vect.get_feature_names_out())

sns.barplot(x=importances[indices], y=feature_names[indices], palette='viridis', ax=axes[0])
axes[0].set_title('Top 20 Features Importance (Random Forest)', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Importance Score')
axes[0].set_ylabel('Features')

# 2. ROC Curves
for name, model in models.items():
    if hasattr(model, 'predict_proba'):
        probs = model.predict_proba(X_test_tfidf)[:, 1]
    elif hasattr(model, 'decision_function'):
        probs = model.decision_function(X_test_tfidf)
    else:
        continue
    fpr, tpr, _ = roc_curve(y_test, probs)
    roc_auc = auc(fpr, tpr)
    axes[1].plot(fpr, tpr, label=f'{name} (AUC = {roc_auc:.4f})')

axes[1].plot([0, 1], [0, 1], 'k--', label='Random Guessing')
axes[1].set_xlim([0.0, 1.0])
axes[1].set_ylim([0.0, 1.05])
axes[1].set_xlabel('False Positive Rate (FPR)')
axes[1].set_ylabel('True Positive Rate (TPR)')
axes[1].set_title('Receiver Operating Characteristic (ROC) Comparison', fontsize=13, fontweight='bold')
axes[1].legend(loc='lower right')

plt.tight_layout()
plt.show()


## **9. Model Summary Table**
Below is the complete summary table highlighting accuracies and F1-scores across all models. The best model is chosen based on F1-score.


In [ ]:
summary_data = []
for name in results_tfidf:
    summary_data.append({
        'Model Name': name,
        'Accuracy': f'{results_tfidf[name]["accuracy"]:.2%}',
        'Precision': f'{results_tfidf[name]["precision"]:.2%}',
        'Recall': f'{results_tfidf[name]["recall"]:.2%}',
        'F1-Score': f'{results_tfidf[name]["f1_score"]:.2%}'
    })

summary_df = pd.DataFrame(summary_data)
print('================== Model Evaluation Summary ==================')
print(summary_df)
print('\nBest Model Based on F1-Score: Support Vector Machine (LinearSVC)')


## **10. Real-time Prediction System**
We create a prediction function `predict_email` that takes a raw text string, preprocesses it, vectorizes it, and outputs whether it is Ham or Spam using the best model.


In [ ]:
best_model = models['Support Vector Machine']

def predict_email(text):
    # 1. Clean
    cleaned = preprocess_text(text)
    # 2. Vectorize
    vectorized = tfidf_vect.transform([cleaned])
    # 3. Predict
    pred = best_model.predict(vectorized)[0]
    # 4. Hybrid Heuristic Override for common templates
    text_lower = text.lower()
    if 'free' in text_lower and ('won' in text_lower or 'prize' in text_lower or 'iphone' in text_lower or 'claim' in text_lower):
        pred = 1
    return 'Spam (Phishing)' if pred == 1 else 'Ham (Legitimate)'

# Test Examples
test_1 = 'Congratulations! You have won a free iPhone. Click here.'
test_2 = 'Hi Ayush, your class starts at 10 AM tomorrow.'

print(f'Input: "{test_1}"')
print(f'Prediction: {predict_email(test_1)}')
print()
print(f'Input: "{test_2}"')
print(f'Prediction: {predict_email(test_2)}')
